# The Six Postulates of Quantum Mechanics — Expanded Edition
### Bridging Linear Algebra, Probability, and Differential Equations

It is a common challenge for students to bridge the notational and conceptual gap between the mathematics they have already learned and the new framework presented in quantum mechanics. This notebook provides a breakdown of David McIntyre's six postulates of quantum mechanics, translated into the language of linear algebra, probability, and differential equations, complete with concrete mathematical examples and Python visualizations.

**Expanded content includes:**
- Reusable `QuantumState` and `QuantumOperator` classes
- Spectral decomposition and projection operators (Postulate 2)
- Expectation values, variance, and the Robertson uncertainty principle
- Degenerate collapse and the projection postulate (Postulate 5)
- Heisenberg picture and Rabi oscillations (Postulate 6)
- Two-qubit tensor products and Bell entanglement
- Density matrix formalism (pure vs. mixed states)

**Reference:** McIntyre, D. H. (2022). *Quantum Mechanics: A Paradigms Approach.* Cambridge University Press.

## Code Preamble: Import Libraries and Helper Functions

First, we import the necessary Python libraries and define helper functions for Bloch sphere visualization, along with reusable `QuantumState` and `QuantumOperator` classes that make the relationship between bras, kets, and operators explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import expm

def plot_bloch_sphere(ax, states=[], labels=[], colors=[], title=""):
    """Plots the Bloch sphere and given state vectors."""
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(xs, ys, zs, color='lightcyan', alpha=0.1)
    ax.set_box_aspect([1, 1, 1])
    for axis in [([[-1,1],[0,0],[0,0]]),
                 ([[0,0],[-1,1],[0,0]]),
                 ([[0,0],[0,0],[-1,1]])]:
        ax.plot(axis[0], axis[1], axis[2], color='gray', linestyle='--', alpha=0.5)
    ax.text(0, 0,  1.15, "|0\u27E9", fontsize=9, ha='center')
    ax.text(0, 0, -1.25, "|1\u27E9", fontsize=9, ha='center')
    ax.text( 1.15, 0, 0,  "|+\u27E9", fontsize=9)
    ax.text(-1.35, 0, 0,  "|-\u27E9", fontsize=9)
    ax.text(0,  1.15, 0,  "|i\u27E9", fontsize=9)
    ax.text(0, -1.35, 0,  "|-i\u27E9", fontsize=9)
    for i, state in enumerate(states):
        alpha, beta = state[0, 0], state[1, 0]
        theta = np.pi if np.abs(alpha) < 1e-9 else 2 * np.arccos(np.clip(np.abs(alpha), 0, 1))
        phi = np.angle(beta) - np.angle(alpha)
        xc = np.sin(theta) * np.cos(phi)
        yc = np.sin(theta) * np.sin(phi)
        zc = np.cos(theta)
        lbl = labels[i] if i < len(labels) else ''
        col = colors[i] if i < len(colors) else 'red'
        ax.quiver(0, 0, 0, xc, yc, zc, color=col, length=1,
                  arrow_length_ratio=0.15, label=lbl, linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    if labels:
        ax.legend(loc='upper left', fontsize=8)


def state_to_bloch(state):
    """Convert a 2-component column state vector to Bloch sphere (x,y,z)."""
    alpha, beta = state[0, 0], state[1, 0]
    theta = np.pi if np.abs(alpha) < 1e-9 else 2 * np.arccos(np.clip(np.abs(alpha), 0, 1))
    phi = np.angle(beta) - np.angle(alpha)
    return np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)

In [ ]:
class QuantumState:
    """
    A normalized complex column vector representing a pure quantum state.

    Mathematical note: This is an element of the projective Hilbert space CP^(n-1).
    For n=2 (qubits), CP^1 is isomorphic to S^2, which is exactly the Bloch sphere.
    Two states differing only by a global phase e^(i*phi) are physically identical.
    """
    def __init__(self, amplitudes):
        arr = np.array(amplitudes, dtype=complex).reshape(-1, 1)
        norm = np.sqrt((arr.conj().T @ arr)[0, 0].real)
        if norm < 1e-12:
            raise ValueError("Cannot normalize the zero vector.")
        self.vec = arr / norm

    @property
    def bra(self):
        return self.vec.conj().T

    def inner(self, other):
        """<self|other> -- complex inner product."""
        return (self.bra @ other.vec)[0, 0]

    def outer(self, other=None):
        """Outer product |self><other| (or |self><self| if other is None)."""
        if other is None:
            other = self
        return self.vec @ other.bra

    def norm(self):
        return np.sqrt(self.inner(self).real)

    def expect(self, operator):
        """<psi|A|psi> -- expectation value of an operator."""
        return (self.bra @ operator @ self.vec)[0, 0]

    def variance(self, operator):
        """Var(A) = <A^2> - <A>^2 -- variance of an observable."""
        exp_A  = self.expect(operator)
        exp_A2 = self.expect(operator @ operator)
        return (exp_A2 - exp_A**2).real

    def prob_of(self, eigenvector):
        """Born rule: P = |<eigenvector|psi>|^2"""
        if isinstance(eigenvector, QuantumState):
            amp = eigenvector.inner(self)
        else:
            amp = (eigenvector.conj().T @ self.vec)[0, 0]
        return np.abs(amp)**2

    def __repr__(self):
        return f"QuantumState(\n{self.vec}\n)"

print("QuantumState class defined.")

In [ ]:
class QuantumOperator:
    """
    A linear operator (matrix) on a Hilbert space.

    For observables, the matrix must be Hermitian: A = A-dagger.
    The spectral theorem guarantees real eigenvalues and an orthonormal
    eigenbasis -- this is the bridge between Hermitian matrices and
    measurable physical quantities.
    """
    def __init__(self, matrix):
        self.mat = np.array(matrix, dtype=complex)

    def is_hermitian(self, tol=1e-10):
        return np.allclose(self.mat, self.mat.conj().T, atol=tol)

    def is_unitary(self, tol=1e-10):
        n = self.mat.shape[0]
        return np.allclose(self.mat @ self.mat.conj().T, np.eye(n), atol=tol)

    def spectral_decomposition(self):
        """
        For a Hermitian operator A, the spectral theorem gives:
            A = sum_n  lambda_n |phi_n><phi_n|
        np.linalg.eigh is used (not eig) because it guarantees real, sorted
        eigenvalues for Hermitian matrices.
        """
        if not self.is_hermitian():
            raise ValueError("Spectral decomposition requires a Hermitian operator.")
        eigenvalues, eigenvectors = np.linalg.eigh(self.mat)
        projectors = [np.outer(eigenvectors[:, i], eigenvectors[:, i].conj())
                      for i in range(len(eigenvalues))]
        return eigenvalues, eigenvectors, projectors

    def commutator(self, other):
        """[A, B] = AB - BA. Zero iff A and B share an eigenbasis."""
        return QuantumOperator(self.mat @ other.mat - other.mat @ self.mat)

    def time_evolve(self, t, hbar=1.0):
        """Unitary time evolution operator U(t) = exp(-i H t / hbar)."""
        return QuantumOperator(expm(-1j * self.mat * t / hbar))

    def __matmul__(self, other):
        if isinstance(other, QuantumOperator):
            return QuantumOperator(self.mat @ other.mat)
        return self.mat @ other

    def __repr__(self):
        return f"QuantumOperator(\n{self.mat}\n)"

print("QuantumOperator class defined.")

In [ ]:
# Standard Operators (Pauli matrices and raising/lowering)
I2   = QuantumOperator([[1, 0], [0, 1]])
Sx   = QuantumOperator([[0,    1   ], [1,    0   ]])
Sy   = QuantumOperator([[0,   -1j  ], [1j,   0   ]])
Sz   = QuantumOperator([[1,    0   ], [0,   -1   ]])
Sp   = QuantumOperator([[0,    1   ], [0,    0   ]])  # raising
Sm   = QuantumOperator([[0,    0   ], [1,    0   ]])  # lowering

# Computational basis and cardinal states
ket0  = QuantumState([1, 0])
ket1  = QuantumState([0, 1])
ket_p = QuantumState([1,  1])   # |+>
ket_m = QuantumState([1, -1])   # |->
ket_i = QuantumState([1,  1j])  # |i>
ket_mi= QuantumState([1, -1j])  # |-i>

print("Standard operators and basis states defined.")
print(f"  Pauli X:\n{Sx.mat.real}\n")
print(f"  Pauli Y:\n{Sy.mat}\n")
print(f"  Pauli Z:\n{Sz.mat.real}")

## Postulate 1: The State of the System

**Quantum Mechanics Statement:** The state of a quantum mechanical system is completely described by a state vector $|\psi\rangle$, which is a vector in a complex vector space known as a Hilbert space.

**Translation into Linear Algebra:**
A quantum state is a unit vector in a complex Hilbert space $\mathcal{H}$. The notation $|\psi\rangle$ ("ket") is a column vector; $\langle\psi|$ ("bra") is its conjugate transpose. Normalization requires $\langle\psi|\psi\rangle = 1$.

**Geometric remark:**
Because a global phase $e^{i\varphi}|\psi\rangle$ is physically indistinguishable from $|\psi\rangle$, the true state space for a qubit is the projective space $\mathbb{CP}^1 \cong S^2$ — the Bloch sphere. The $U(1)$ fiber of global phases is quotiented out, leaving a 2-real-dimensional manifold.

**Concrete Example:**
$$
|\psi\rangle = \frac{1}{\sqrt{5}}\begin{pmatrix} 1 \\ 2i \end{pmatrix}, \quad
\langle\psi| = \frac{1}{\sqrt{5}}\begin{pmatrix} 1 & -2i \end{pmatrix}
$$
$$
\langle\psi|\psi\rangle = \frac{1}{5}(1\cdot 1 + (-2i)(2i)) = \frac{1}{5}(1 + 4) = 1
$$

In [ ]:
# POSTULATE 1: The State of the System
psi1 = QuantumState([1/np.sqrt(5), 2j/np.sqrt(5)])
print(f"State |psi> = (1/sqrt5)|0> + (2i/sqrt5)|1>:\n{psi1.vec}\n")
print(f"<psi|psi> = {psi1.inner(psi1):.6f}  (should be 1)\n")

# Global vs relative phase demonstration
psi1_global = QuantumState([1j/np.sqrt(5), -2/np.sqrt(5)])  # e^(i*pi/2)|psi>
psi1_relative = QuantumState([1/np.sqrt(5), -2j/np.sqrt(5)])

print("Global phase check:")
print(f"  |<psi_orig|psi_global>|^2 = {psi1.prob_of(psi1_global):.6f}  (should be 1 -- same physical state)")
print(f"  |<psi_orig|psi_relative>|^2 = {psi1.prob_of(psi1_relative):.6f}  (< 1 -- different state)")

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
plot_bloch_sphere(ax, [psi1.vec], ["|psi>"], ["steelblue"],
                  "Postulate 1: A Quantum State on the Bloch Sphere")
plt.tight_layout()
plt.show()

## Postulate 2: Observables and Operators

**Quantum Mechanics Statement:** Every measurable physical quantity, known as an observable, is represented by a linear, Hermitian operator that acts on the state vectors in the Hilbert space.

**Translation into Linear Algebra:**
Every observable $A$ is a Hermitian (self-adjoint) linear operator: $A = A^\dagger$. The spectral theorem guarantees:
1. All eigenvalues are real $\rightarrow$ physical measurement results
2. Eigenvectors of distinct $\lambda$ are orthogonal $\rightarrow$ mutually exclusive outcomes
3. Eigenvectors form a complete basis $\rightarrow$ any state can be expanded in them

**Spectral decomposition:**
$$A = \sum_n \lambda_n P_n, \quad \text{where } P_n = |\phi_n\rangle\langle\phi_n|$$
is the projection operator onto the $n$-th eigenspace.

**Expectation value:**
$$\langle A\rangle_\psi = \langle\psi|A|\psi\rangle = \sum_n \lambda_n |\langle\phi_n|\psi\rangle|^2$$

**Commutators and compatibility:**
$[A, B] = AB - BA$. If $[A, B] = 0$, $A$ and $B$ share an eigenbasis and are simultaneously measurable ("compatible observables"). The Pauli matrices satisfy $[\sigma_i, \sigma_j] = 2i\varepsilon_{ijk}\sigma_k$, so no two are compatible.

In [ ]:
# POSTULATE 2: Observables and Operators
for name, op in [("sigma_x", Sx), ("sigma_y", Sy), ("sigma_z", Sz)]:
    print(f"{name}:  Hermitian = {op.is_hermitian()}")

print()
comm_xy = Sx.commutator(Sy)
print(f"[sigma_x, sigma_y] =\n{comm_xy.mat}")
print(f"\n2i*sigma_z =\n{2j * Sz.mat}\n")

# Spectral decomposition of sigma_z
eigenvalues_z, eigenvectors_z, projectors_z = Sz.spectral_decomposition()
print("Spectral decomposition of sigma_z:")
for i, (lam, P) in enumerate(zip(eigenvalues_z, projectors_z)):
    print(f"  lambda_{i} = {lam:.0f},  P_{i} =\n{P.real}")

# Expectation value and variance
print(f"\nFor |psi> = {psi1.vec.T}:")
exp_z  = psi1.expect(Sz.mat)
var_z  = psi1.variance(Sz.mat)
print(f"  <sigma_z> = {exp_z.real:.4f}")
print(f"  Var(sigma_z) = <sigma_z^2> - <sigma_z>^2 = {var_z:.4f}")
print(f"  std(sigma_z) = {np.sqrt(var_z):.4f}")

In [ ]:
# Visualize the action of Pauli operators
for op, name, color, init in [(Sx, "sigma_x acts on |0>", "darkorange", ket0),
                               (Sz, "sigma_z acts on |+>", "purple", ket_p)]:
    final_vec = op.mat @ init.vec
    final_state = QuantumState(final_vec)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    plot_bloch_sphere(ax, [init.vec, final_state.vec],
                     ["Initial", "After operator"], ["green", color], name)
    plt.tight_layout()
    plt.show()

## Postulate 3: Possible Outcomes of a Measurement

**Quantum Mechanics Statement:** The only possible result of a measurement of an observable is one of the eigenvalues of the corresponding operator.

**Translation into Linear Algebra:**
The only possible results when measuring observable $A$ are the eigenvalues $\{\lambda_n\}$ of the corresponding Hermitian operator. For Hermitian operators these are guaranteed real by the spectral theorem.

**Discrete vs. continuous spectra:**
Finite-dimensional systems (like qubits) have discrete spectra. For operators like $\hat{X}$ or $\hat{P}$ acting on $L^2(\mathbb{R})$, the spectrum is continuous and the eigenstates are not normalizable in the usual sense (they are Dirac $\delta$-distributions). In that case the Born rule uses probability *density*: $dP = |\psi(x)|^2 dx$.

**Degenerate eigenvalues:**
If multiple eigenvectors share the same eigenvalue $\lambda$ (degenerate case), any vector in that eigenspace is a valid post-measurement state. The appropriate projector is $P_\lambda = \sum_{n \in \lambda} |\phi_n\rangle\langle\phi_n|$.

In [ ]:
# POSTULATE 3: Possible Outcomes of a Measurement

# Demonstrate with a degenerate 3x3 operator
H_degen = QuantumOperator([[1, 0, 0], [0, 1, 0], [0, 0, -1]])  # lambda=1 doubly degenerate
evals, evecs, projs = H_degen.spectral_decomposition()
print("3x3 operator with degenerate spectrum:")
print(f"  Eigenvalues: {evals}")
print(f"  Distinct outcomes: {np.unique(evals.round(10))}\n")

# Standard qubit example
psi3 = QuantumState([1/np.sqrt(5), 2/np.sqrt(5)])
print(f"Measuring sigma_z on |psi> = (1/sqrt5)|0> + (2/sqrt5)|1>:")
for lam, evec in zip(eigenvalues_z, eigenvectors_z.T):
    p = psi3.prob_of(evec.reshape(-1, 1))
    print(f"  P(lambda={lam:+.0f}) = {p:.4f}")

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
plot_bloch_sphere(ax,
                  [psi3.vec, eigenvectors_z[:, 0].reshape(-1, 1),
                             eigenvectors_z[:, 1].reshape(-1, 1)],
                  ["|psi>", "|0> (lambda=+1)", "|1> (lambda=-1)"],
                  ["steelblue", "black", "dimgray"],
                  "Postulate 3: State and Eigenvectors of sigma_z")
plt.tight_layout()
plt.show()

## Postulate 4: Born Rule — Probability of Measurement Outcomes

**Quantum Mechanics Statement:** The probability of obtaining a specific eigenvalue $a_n$ in a measurement of the observable $A$ on a system in the state $|\psi\rangle$ is given by:
$$P(\lambda_n) = |\langle\phi_n|\psi\rangle|^2 = \langle\psi|P_n|\psi\rangle$$

The second form makes the geometry clear: probability is the squared length of the projection of $|\psi\rangle$ onto the eigenspace.

**Expectation value from Born rule:**
$$\langle A\rangle = \sum_n \lambda_n P(\lambda_n) = \langle\psi|A|\psi\rangle$$

This connects Born rule probability to linear algebra: the expectation value is simply the inner product $\langle\psi|A|\psi\rangle$, a weighted average over the spectral decomposition.

**Completeness check:**
$$\sum_n P(\lambda_n) = \sum_n \langle\psi|P_n|\psi\rangle = \langle\psi|\left(\sum_n P_n\right)|\psi\rangle = \langle\psi|I|\psi\rangle = 1$$

In [ ]:
# POSTULATE 4: Born Rule

psi4 = QuantumState([1/np.sqrt(5), 2j/np.sqrt(5)])
print(f"State |psi>:\n{psi4.vec}\n")

print("Measuring sigma_x on |psi>:")
evals_x, evecs_x, projs_x = Sx.spectral_decomposition()
probs_x = []
for lam, P in zip(evals_x, projs_x):
    prob = (psi4.bra @ P @ psi4.vec)[0, 0].real
    probs_x.append(prob)
    print(f"  P(lambda={lam:+.0f}) = <psi|P_n|psi> = {prob:.4f}")

print(f"\nProbabilities sum to: {sum(probs_x):.6f}")
print(f"<sigma_x> = {psi4.expect(Sx.mat).real:.4f}")
print(f"<sigma_y> = {psi4.expect(Sy.mat).real:.4f}")
print(f"<sigma_z> = {psi4.expect(Sz.mat).real:.4f}")

# Visualization: probability bar charts for sigma_x and sigma_z
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(12, 5))

# sigma_x probabilities
probs_x_plot = [(psi4.bra @ P @ psi4.vec)[0, 0].real for P in projs_x]
ax_l.bar([f"lambda={v:+.0f}" for v in evals_x], probs_x_plot,
         color=['steelblue', 'salmon'])
ax_l.set_ylim(0, 1); ax_l.set_ylabel("Probability")
ax_l.set_title("Born Rule: Measuring sigma_x")

# sigma_z probabilities
evals_z_plot, _, projs_z_plot = Sz.spectral_decomposition()
probs_z_plot = [(psi4.bra @ P @ psi4.vec)[0, 0].real for P in projs_z_plot]
ax_r.bar([f"lambda={v:+.0f}" for v in evals_z_plot], probs_z_plot,
         color=['steelblue', 'salmon'])
ax_r.set_ylim(0, 1); ax_r.set_ylabel("Probability")
ax_r.set_title("Born Rule: Measuring sigma_z")

plt.tight_layout()
plt.show()

## Postulate 5: State Collapse (Projection Postulate)

**Quantum Mechanics Statement:** If a measurement of an observable $A$ on a system in the state $|\psi\rangle$ yields the result $\lambda_n$, the state of the system immediately after the measurement collapses to the normalized projection onto the $\lambda_n$ eigenspace:

$$|\psi\rangle \;\rightarrow\; \frac{P_n|\psi\rangle}{\sqrt{\langle\psi|P_n|\psi\rangle}}$$

For a non-degenerate eigenvalue this is just $|\phi_n\rangle$. For a degenerate eigenvalue, $P_n$ projects onto a subspace; the post-measurement state retains the component of $|\psi\rangle$ within that subspace.

**Immediate re-measurement:** After collapse, $P(\lambda_n \text{ again}) = \langle\psi_\text{after}|P_n|\psi_\text{after}\rangle = 1$.

**Note:** This "projection postulate" (von Neumann) is the standard treatment. More general measurements are described by Positive Operator-Valued Measures (POVMs), where the measurement operators $M_k$ satisfy $\sum M_k^\dagger M_k = I$ but are not necessarily projectors.

In [ ]:
# POSTULATE 5: State Collapse

# Simulate repeated measurements -- Monte Carlo verification
evals_z, evecs_z, projs_z = Sz.spectral_decomposition()
probs_z = [(psi4.bra @ P @ psi4.vec)[0, 0].real for P in projs_z]

rng = np.random.default_rng(42)
N_trials = 10000
outcomes = rng.choice(evals_z, size=N_trials, p=probs_z)
print(f"Monte Carlo ({N_trials} trials) of measuring sigma_z on |psi>:")
for lam, prob in zip(evals_z, probs_z):
    freq = np.mean(outcomes == lam)
    print(f"  lambda={lam:+.0f}:  theoretical P={prob:.4f},  observed freq={freq:.4f}")

# Single measurement -> collapse -> re-measure
outcome_idx = rng.choice(len(evals_z), p=probs_z)
collapsed_vec = projs_z[outcome_idx] @ psi4.vec
collapsed_vec /= np.linalg.norm(collapsed_vec)
collapsed_state = QuantumState(collapsed_vec)
print(f"\nSingle measurement outcome: lambda = {evals_z[outcome_idx]:+.0f}")
print(f"Collapsed state:\n{collapsed_state.vec}")
probs_after = [(collapsed_state.bra @ P @ collapsed_state.vec)[0, 0].real
               for P in projs_z]
print(f"P(same outcome on re-measurement) = {probs_after[outcome_idx]:.6f}  (should be 1)")

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
plot_bloch_sphere(ax,
                  [psi4.vec, collapsed_state.vec],
                  ["Before: |psi>", f"After: lambda={evals_z[outcome_idx]:+.0f} collapse"],
                  ["steelblue", "crimson"],
                  "Postulate 5: Collapse to Eigenstate")
plt.tight_layout()
plt.show()

## Postulate 6: Time Evolution — Schrödinger Equation

**Quantum Mechanics Statement:** The time evolution of a quantum system is governed by the Schrödinger equation:
$$i\hbar\frac{d}{dt}|\psi(t)\rangle = H|\psi(t)\rangle$$

**Translation into Differential Equations and Linear Algebra:**
With constant $H$, the formal solution is the unitary operator:
$$|\psi(t)\rangle = U(t)|\psi(0)\rangle, \quad U(t) = e^{-iHt/\hbar}$$

$U(t)$ is unitary ($U^\dagger U = I$), so it preserves the norm of $|\psi\rangle$. Geometrically, it rotates the state vector — on the Bloch sphere, $H = \omega\hat{n}\cdot\vec{\sigma}/2$ generates precession around the axis $\hat{n}$ at angular frequency $\omega$.

In the energy eigenbasis ($H|E_n\rangle = E_n|E_n\rangle$):
$$|\psi(t)\rangle = \sum_n c_n e^{-iE_n t/\hbar} |E_n\rangle \quad \text{where } c_n = \langle E_n|\psi(0)\rangle$$

**Heisenberg picture:** Equivalently, operators evolve and states stay fixed:
$$A_H(t) = U^\dagger(t) A_S U(t), \quad \frac{d}{dt}A_H = \frac{i}{\hbar}[H, A_H]$$

Both pictures give identical expectation values and probabilities.

In [ ]:
# POSTULATE 6: Time Evolution

omega = 2.0  # Larmor frequency
H_larmor = QuantumOperator(omega * Sz.mat)  # Precession around Z
psi6 = QuantumState([1, 1])  # |+>, pointing along X

times = np.linspace(0, 2*np.pi/omega, 200)
states_t = []
exp_x, exp_y, exp_z_t = [], [], []

for t in times:
    U = H_larmor.time_evolve(t)
    psi_t = QuantumState(U.mat @ psi6.vec)
    states_t.append(psi_t)
    exp_x.append(psi_t.expect(Sx.mat).real)
    exp_y.append(psi_t.expect(Sy.mat).real)
    exp_z_t.append(psi_t.expect(Sz.mat).real)

# Plot expectation value precession
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, exp_x, label="<sigma_x>", color='steelblue')
ax.plot(times, exp_y, label="<sigma_y>", color='darkorange')
ax.plot(times, exp_z_t, label="<sigma_z>", color='green', linestyle='--')
ax.set_xlabel("Time t (hbar = 1)")
ax.set_ylabel("Expectation value")
ax.set_title("Postulate 6: Larmor Precession  H = omega*sigma_z\n"
             "<sigma_x> and <sigma_y> oscillate; <sigma_z> is conserved (commutes with H)")
ax.legend(); ax.set_ylim(-1.1, 1.1); ax.axhline(0, color='k', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3D trajectory on Bloch sphere
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
path_x, path_y, path_z_path = zip(*[state_to_bloch(s.vec) for s in states_t])
plot_bloch_sphere(ax, [psi6.vec], ["|psi(0)> = |+>"], ["green"],
                  "Postulate 6: Precession Trajectory on Bloch Sphere")
ax.plot(path_x, path_y, path_z_path, color='purple', linewidth=1.5,
        label="Path of |psi(t)>", alpha=0.8)
ax.legend()
plt.tight_layout()
plt.show()

# Heisenberg picture comparison
print("Heisenberg picture vs Schrodinger picture at t = pi/(2*omega):")
t_check = np.pi / (2 * omega)
U_check = H_larmor.time_evolve(t_check)
psi_t_check = QuantumState(U_check.mat @ psi6.vec)

# Schrodinger: state evolves, operator fixed
exp_sx_schrodinger = psi_t_check.expect(Sx.mat).real

# Heisenberg: operator evolves, state fixed
Sx_H = U_check.mat.conj().T @ Sx.mat @ U_check.mat
exp_sx_heisenberg = psi6.expect(Sx_H).real

print(f"  Schrodinger: <psi(t)|sigma_x|psi(t)> = {exp_sx_schrodinger:.6f}")
print(f"  Heisenberg:  <psi(0)|sigma_x(t)|psi(0)> = {exp_sx_heisenberg:.6f}")
print(f"  Identical: {np.isclose(exp_sx_schrodinger, exp_sx_heisenberg)}")

## Bonus 1: Robertson Uncertainty Principle

The Heisenberg uncertainty principle is a consequence of the commutator structure of quantum mechanics, not a statement about measurement disturbance.

**Robertson's theorem:**
$$\Delta A \cdot \Delta B \geq \frac{1}{2}|\langle[A, B]\rangle|$$

where $\Delta A = \sqrt{\langle A^2\rangle - \langle A\rangle^2}$ is the standard deviation of $A$.

For position and momentum: $[\hat{X}, \hat{P}] = i\hbar \;\rightarrow\; \Delta X \cdot \Delta P \geq \hbar/2$

For spin-1/2: $[\sigma_x, \sigma_y] = 2i\sigma_z \;\rightarrow\; \Delta\sigma_x \cdot \Delta\sigma_y \geq |\langle\sigma_z\rangle|$

The bound is *state-dependent* through $\langle\sigma_z\rangle$ — it is tight only for coherent spin states.

In [ ]:
# BONUS 1: Robertson Uncertainty Principle

# Sweep theta on the Bloch sphere and verify Robertson bound
thetas = np.linspace(0, np.pi, 100)
delta_x_arr, delta_y_arr, bound_arr = [], [], []

for theta in thetas:
    psi_theta = QuantumState([np.cos(theta/2), np.sin(theta/2)])
    dX = np.sqrt(max(psi_theta.variance(Sx.mat), 0))
    dY = np.sqrt(max(psi_theta.variance(Sy.mat), 0))
    exp_comm = psi_theta.expect(Sx.commutator(Sy).mat)  # = 2i<sigma_z>
    bound = 0.5 * np.abs(exp_comm)
    delta_x_arr.append(dX)
    delta_y_arr.append(dY)
    bound_arr.append(bound)

product = np.array(delta_x_arr) * np.array(delta_y_arr)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.degrees(thetas), product,    label="d_sigma_x * d_sigma_y", color='steelblue', linewidth=2)
ax.plot(np.degrees(thetas), bound_arr,  label="|<[sigma_x,sigma_y]>|/2 (Robertson bound)",
        color='crimson', linestyle='--', linewidth=2)
ax.fill_between(np.degrees(thetas), bound_arr, product, alpha=0.15, color='steelblue')
ax.set_xlabel("Polar angle theta on Bloch sphere (degrees)")
ax.set_ylabel("Value")
ax.set_title("Robertson Uncertainty Principle: d_sigma_x * d_sigma_y >= |<[sigma_x, sigma_y]>|/2")
ax.legend()
plt.tight_layout()
plt.show()

all_satisfied = np.all(product >= np.array(bound_arr) - 1e-10)
print(f"Robertson bound satisfied for all theta? {all_satisfied}")

## Bonus 2: Entanglement and Tensor Products (Two-Qubit Systems)

For composite systems, the Hilbert space is the tensor product:
$$\mathcal{H} = \mathcal{H}_A \otimes \mathcal{H}_B$$

A two-qubit state lives in $\mathbb{C}^2 \otimes \mathbb{C}^2 \cong \mathbb{C}^4$. The computational basis is: $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.

A state $|\psi\rangle \in \mathcal{H}_A \otimes \mathcal{H}_B$ is **separable** if it can be written as $|a\rangle \otimes |b\rangle$. Otherwise it is **entangled**.

**Bell states — maximally entangled orthonormal basis:**
$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}, \quad |\Phi^-\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}}$$
$$|\Psi^+\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}, \quad |\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}$$

**Entanglement detection via partial trace:**
For a pure state $|\psi\rangle$, compute the reduced density matrix $\rho_A = \text{Tr}_B(|\psi\rangle\langle\psi|)$. If $\rho_A$ is mixed ($\text{Tr}(\rho_A^2) < 1$), the state is entangled. Maximally entangled $\leftrightarrow$ $\rho_A = I/2$ $\leftrightarrow$ $\text{Tr}(\rho_A^2) = 1/2$.

In [ ]:
# BONUS 2: Entanglement and Tensor Products

# Construct Bell states as 4-component vectors in C^4
# Basis ordering: |00>=e0, |01>=e1, |10>=e2, |11>=e3
bell_Phi_plus  = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
bell_Phi_minus = np.array([1, 0, 0,-1], dtype=complex) / np.sqrt(2)
bell_Psi_plus  = np.array([0, 1, 1, 0], dtype=complex) / np.sqrt(2)
bell_Psi_minus = np.array([0, 1,-1, 0], dtype=complex) / np.sqrt(2)

def partial_trace_B(psi_AB, dim_A=2, dim_B=2):
    """Compute reduced density matrix rho_A = Tr_B(|psi><psi|)."""
    rho_AB = np.outer(psi_AB, psi_AB.conj())
    rho_reshaped = rho_AB.reshape(dim_A, dim_B, dim_A, dim_B)
    return np.einsum('ibjb->ij', rho_reshaped)

def purity(rho):
    """Tr(rho^2) -- equals 1 for pure states, 1/d for maximally mixed."""
    return np.trace(rho @ rho).real

# Separable state for comparison
sep_state = np.kron(np.array([1, 0]), np.array([1, 1])/np.sqrt(2))  # |0> x |+>

print(f"{'State':<12}  {'Tr(rho_A^2)':<12}  {'Entangled?'}")
print("-" * 40)
for name, state in [("Separable", sep_state), ("|Phi+>", bell_Phi_plus),
                    ("|Phi->", bell_Phi_minus), ("|Psi+>", bell_Psi_plus)]:
    rho_A = partial_trace_B(state)
    pur   = purity(rho_A)
    entangled = pur < 0.999
    print(f"{name:<12}  {pur:<12.4f}  {entangled}")

# Orthonormality of Bell states
bell_states = [bell_Phi_plus, bell_Phi_minus, bell_Psi_plus, bell_Psi_minus]
bell_names  = ["|Phi+>", "|Phi->", "|Psi+>", "|Psi->"]
gram = np.array([[np.abs(b1 @ b2.conj()) for b2 in bell_states] for b1 in bell_states])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(gram, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(bell_names); ax.set_yticklabels(bell_names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{gram[i,j]:.1f}", ha='center', va='center', fontsize=12)
ax.set_title("|<Bell_i|Bell_j>|  (should be identity)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Bonus 3: Density Matrix Formalism

A pure state $|\psi\rangle$ is an idealization. In practice, quantum systems are often in a **statistical mixture** of pure states — e.g., an ensemble from a thermal source, or a subsystem of an entangled pair.

The density matrix $\rho$ encodes both cases:
- **Pure state:** $\rho = |\psi\rangle\langle\psi|$, $\text{Tr}(\rho) = 1$, $\text{Tr}(\rho^2) = 1$
- **Mixed state:** $\rho = \sum_k p_k |\psi_k\rangle\langle\psi_k|$, $\text{Tr}(\rho) = 1$, $\text{Tr}(\rho^2) < 1$

**Validity conditions:** $\rho = \rho^\dagger$, $\text{Tr}(\rho) = 1$, $\rho \geq 0$ (positive semidefinite)

**Expectation values:** $\langle A\rangle = \text{Tr}(\rho A)$

**Born rule (general):** $P(\lambda_n) = \text{Tr}(\rho P_n)$

**Time evolution:** $\frac{d\rho}{dt} = -\frac{i}{\hbar}[H, \rho]$ (von Neumann equation)

**Von Neumann entropy:** $S = -\text{Tr}(\rho \log \rho) = -\sum_k \lambda_k \log \lambda_k$ measures the mixedness: $S=0$ for pure states, $S=\log(d)$ for maximally mixed.

In [ ]:
# BONUS 3: Density Matrix Formalism

def density_matrix(state_vec):
    """rho = |psi><psi| for a pure state."""
    v = state_vec.reshape(-1, 1)
    return v @ v.conj().T

def von_neumann_entropy(rho):
    """S = -Tr(rho log rho), using eigenvalues of rho."""
    evals = np.linalg.eigvalsh(rho)
    evals = evals[evals > 1e-15]
    return -np.sum(evals * np.log(evals))

# Pure state density matrix
rho_pure  = density_matrix(psi4.vec)
# Mixed state: 50/50 mixture of |0> and |1>
rho_mixed = 0.5 * density_matrix(ket0.vec) + 0.5 * density_matrix(ket1.vec)
# Maximally mixed state
rho_max   = np.eye(2) / 2

for name, rho in [("Pure |psi>", rho_pure), ("Mixed 50/50", rho_mixed),
                  ("Maximally mixed I/2", rho_max)]:
    pur = np.trace(rho @ rho).real
    S   = von_neumann_entropy(rho)
    exp_z_rho = np.trace(rho @ Sz.mat).real
    print(f"{name:<22}  Tr(rho^2)={pur:.4f}  S={S:.4f}  <sigma_z>={exp_z_rho:.4f}")

In [ ]:
# Time evolution of density matrix (von Neumann equation)
# rho(t) = U(t) rho(0) U-dagger(t)
rho0 = density_matrix(ket_p.vec)
exp_x_rho, exp_y_rho = [], []
for t in times:
    U = H_larmor.time_evolve(t).mat
    rho_t = U @ rho0 @ U.conj().T
    exp_x_rho.append(np.trace(rho_t @ Sx.mat).real)
    exp_y_rho.append(np.trace(rho_t @ Sy.mat).real)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times, exp_x_rho, label="Tr(rho(t) sigma_x)", color='steelblue', linewidth=2)
ax.plot(times, exp_y_rho, label="Tr(rho(t) sigma_y)", color='darkorange', linewidth=2)
ax.set_xlabel("Time t"); ax.set_ylabel("Expectation value")
ax.set_title("Von Neumann Equation: rho(t) = U(t) rho(0) U-dagger(t)")
ax.legend(); ax.set_ylim(-1.1, 1.1)
plt.tight_layout()
plt.show()

# Visualize density matrices
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax_i, (name, rho) in zip(axes, [("Pure |psi>", rho_pure),
                                     ("Mixed 50/50", rho_mixed),
                                     ("Maximally mixed", rho_max)]):
    im = ax_i.imshow(np.abs(rho), cmap='viridis', vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax_i.text(j, i, f"{np.abs(rho[i,j]):.2f}", ha='center', va='center',
                     color='white', fontsize=14)
    ax_i.set_title(f"|rho| -- {name}")
    ax_i.set_xticks([0,1]); ax_i.set_yticks([0,1])
    ax_i.set_xticklabels(["|0>", "|1>"]); ax_i.set_yticklabels(["<0|", "<1|"])
    plt.colorbar(im, ax=ax_i)
plt.suptitle("Density Matrices: |rho_ij|  (off-diagonal = quantum coherence)")
plt.tight_layout()
plt.show()

## Bonus 4: Rabi Oscillations — Driven Two-Level System

A driven two-level system has Hamiltonian (in the rotating frame):
$$H_\text{Rabi} = \frac{\Delta}{2}\sigma_z + \frac{\Omega_R}{2}\sigma_x$$

where $\Delta = \omega_\text{laser} - \omega_0$ is the detuning and $\Omega_R$ is the Rabi frequency.

At resonance ($\Delta = 0$): $H = (\Omega_R/2)\sigma_x$. This rotates the Bloch vector around the X-axis at angular frequency $\Omega_R$. Starting from $|0\rangle$, the system oscillates between $|0\rangle$ and $|1\rangle$ with period $T_\text{Rabi} = 2\pi/\Omega_R$.

**Physical realizations:** NMR spin flip, atomic transition driven by laser, superconducting qubit gate operations.

In [ ]:
# BONUS 4: Rabi Oscillations

Omega_R = 2.0
detuning = 0.5

plt.figure(figsize=(10, 5))
for delta, label, color in [(0, "On resonance (delta=0)", "steelblue"),
                             (detuning, f"Off resonance (delta={detuning})", "darkorange")]:
    H_rabi = QuantumOperator((delta/2)*Sz.mat + (Omega_R/2)*Sx.mat)
    psi_rabi0 = QuantumState([1, 0])  # Start in |0>
    t_rabi = np.linspace(0, 4*np.pi/Omega_R, 300)
    pop1 = []
    for t in t_rabi:
        U = H_rabi.time_evolve(t)
        psi_t = QuantumState(U.mat @ psi_rabi0.vec)
        P1 = psi_t.prob_of(ket1.vec)
        pop1.append(P1)
    plt.plot(t_rabi * Omega_R / np.pi, pop1, label=label, color=color, linewidth=2)

plt.xlabel("Time (units of pi/Omega_R)")
plt.ylabel("P(|1>) -- Population of excited state")
plt.title("Rabi Oscillations: Driven Two-Level System")
plt.legend()
plt.ylim(-0.05, 1.05)
plt.axhline(0.5, color='k', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

## Summary: Mathematical Connections

| Postulate | Mathematical Tool | Key Insight |
|-----------|-------------------|-------------|
| **P1** | Complex vector spaces, $\mathbb{CP}^n$ | State $\leftrightarrow$ ray in Hilbert space |
| **P2** | Hermitian matrices, spectral decomposition | Spectral theorem $\rightarrow$ real eigenvalues; commutator $\rightarrow$ compatibility |
| **P3** | Eigenvalue equation | Only eigenvalues are measurable |
| **P4** | Inner products, projectors | Probability $= |\text{projection}|^2$ |
| **P5** | Projection onto subspace | Collapse $=$ conditional state update |
| **P6** | Matrix exponential, ODE | Unitary $\leftrightarrow$ norm-preserving flow |

**Beyond the postulates:**
- **Uncertainty:** Cauchy-Schwarz on Hilbert space $\rightarrow$ $\Delta A \cdot \Delta B \geq \frac{1}{2}|\langle[A,B]\rangle|$
- **Entanglement:** Tensor products, partial trace, Schmidt decomposition
- **Mixed states:** Density matrices, von Neumann entropy, open systems
- **General measurements:** POVMs, Kraus operators, quantum channels

**Reference:** McIntyre, D. H. (2022). *Quantum Mechanics: A Paradigms Approach.* Cambridge University Press.